In [1]:
import os
import torch
import glob
import cv2
import numpy as np
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image

class YoloDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.img_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        self.label_files = sorted(glob.glob(os.path.join(label_dir, "*.txt")))
        self.class_map = {0: "Background", 1: "Chair", 2: "Person", 3: "Table"}  # Change this as needed

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        label_path = self.label_files[idx]

        image = Image.open(img_path).convert("RGB")
        w, h = image.size  # Image width and height

        # Read annotation file
        boxes = []
        labels = []
        with open(label_path, "r") as file:
            for line in file.readlines():
                class_id, x_center, y_center, width, height = map(float, line.strip().split())

                # Convert YOLO format (normalized) to Faster R-CNN format (absolute)
                x_min = (x_center - width / 2) * w
                y_min = (y_center - height / 2) * h
                x_max = (x_center + width / 2) * w
                y_max = (y_center + height / 2) * h

                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(int(class_id) + 1)  # Shift class IDs by 1 (0 is background in FRCNN)

        # Convert to torch tensors
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self.transform:
            image = self.transform(image)

        return image, target

# Define dataset paths
train_dataset = YoloDataset(
    img_dir="D:/split_2/train/images",
    label_dir="D:/split_2/train/labels",
    transform=transforms.ToTensor()
)

val_dataset = YoloDataset(
    img_dir="D:/split_2/val/images",
    label_dir="D:/split_2/val/labels",
    transform=transforms.ToTensor()
)

# DataLoader
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))



c:\Users\jesli\anaconda3\envs\pv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Function to load the Faster R-CNN model
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
num_classes = 4  # Background + chair, human, table
model = get_model(num_classes).to(device)

# Define optimizer and LR scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training function
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch}] Loss: {loss.item():.4f}")

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    train_one_epoch(model, optimizer, train_loader, device, epoch)
    lr_scheduler.step()
    
    # Save model
    torch.save(model.state_dict(), f"fasterrcnn_resnet50_epoch_{epoch + 1}.pth")
    print(f"Model saved: fasterrcnn_resnet50_epoch_{epoch + 1}.pth")


In [ ]:

import os
import torch
import glob
import cv2
import numpy as np
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from PIL import Image

class YoloDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.img_files = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
        self.label_files = sorted(glob.glob(os.path.join(label_dir, "*.txt")))
        self.class_map = {0: "Rice Leaf Roller",1: "Rice Leaf Caterpillar",2: "Paddy Stem Maggot",3: "Asiatic Rice Borer",4: "Yellow Rice Borer",5: "Rice Gall Midge",6: "Rice Stemfly",7: "Brown Plant Hopper",8: "White Backed Plant Hopper",9: "Small Brown Plant Hopper",10: "Rice Water Weevil",11: "Rice Leafhopper",12: "Grain Spreader Thrips",13: "Rice Shell Pest" }  # Change this as needed

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        label_path = self.label_files[idx]

        image = Image.open(img_path).convert("RGB")
        w, h = image.size  # Image width and height

        # Read annotation file
        boxes = []
        labels = []
        with open(label_path, "r") as file:
            for line in file.readlines():
                class_id, x_center, y_center, width, height = map(float, line.strip().split())

                # Convert YOLO format (normalized) to Faster R-CNN format (absolute)
                x_min = (x_center - width / 2) * w
                y_min = (y_center - height / 2) * h
                x_max = (x_center + width / 2) * w
                y_max = (y_center + height / 2) * h

                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(int(class_id) + 1)  # Shift class IDs by 1 (0 is background in FRCNN)

        # Convert to torch tensors
        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self.transform:
            image = self.transform(image)

        return image, target

# Define dataset paths
train_dataset = YoloDataset(
    img_dir="D:/split_2/train/images",
    label_dir="D:/split_2/train/labels",
    transform=transforms.ToTensor()
)

val_dataset = YoloDataset(
    img_dir="D:/split_2/val/images",
    label_dir="D:/split_2/val/labels",
    transform=transforms.ToTensor()
)

# DataLoader
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))


import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Function to load the Faster R-CNN model
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
num_classes = 14  # Background + chair, human, table
model = get_model(num_classes).to(device)

# Define optimizer and LR scheduler
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Training function
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        # Forward pass
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch}] Loss: {loss.item():.4f}")

# Training loop
num_epochs = 5
for epoch in range(num_epochs):
    train_one_epoch(model, optimizer, train_loader, device, epoch)
    lr_scheduler.step()
    
    # Save model
    torch.save(model.state_dict(), f"fasterrcnn_resnet50_epoch_{epoch + 1}.pth")
    print(f"Model saved: fasterrcnn_resnet50_epoch_{epoch + 1}.pth")


'pwd' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# ✅ Step 1: Import Required Libraries
import torch
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image
import glob
import random
import os

# ✅ Step 2: Define Class Labels
CLASS_MAP = {
    0: "Rice Leaf Roller",
    1: "Rice Leaf Caterpillar",
    2: "Paddy Stem Maggot",
    3: "Asiatic Rice Borer",
    4: "Yellow Rice Borer",
    5: "Rice Gall Midge",
    6: "Rice Stemfly",
    7: "Brown Plant Hopper",
    8: "White Backed Plant Hopper",
    9: "Small Brown Plant Hopper",
    10: "Rice Water Weevil",
    11: "Rice Leafhopper",
    12: "Grain Spreader Thrips",
    13: "Rice Shell Pest"
}

NUM_CLASSES = len(CLASS_MAP)  # 14 pest classes

# ✅ Step 3: Load the Trained Model
def get_model(num_classes):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load trained model
model = get_model(NUM_CLASSES).to(device)
model.load_state_dict(torch.load("fasterrcnn_resnet50_final.pth", map_location=device))
model.eval()  # Set model to evaluation mode

print("✅ Model loaded successfully!")

# ✅ Step 4: Select Random Test Images
test_images_path = "D:/split_2/test/images"
all_test_images = glob.glob(os.path.join(test_images_path, "*.*"))
selected_images = random.sample(all_test_images, 5)  # Select 5 random images

# ✅ Step 5: Define Function to Run Inference
def run_inference(image_path):
    # Load image
    image = Image.open(image_path).convert("RGB")
    orig_w, orig_h = image.size

    # Transform image
    transform = transforms.ToTensor()
    image_tensor = transform(image).unsqueeze(0).to(device)

    # Run inference
    with torch.no_grad():
        prediction = model(image_tensor)

    return image, prediction[0]

# ✅ Step 6: Define Function to Plot Predictions
def draw_boxes(image, prediction, score_threshold=0.5):
    plt.figure(figsize=(8, 6))
    plt.imshow(image)
    ax = plt.gca()

    for i in range(len(prediction["boxes"])):
        score = prediction["scores"][i].item()
        if score < score_threshold:
            continue  # Skip low-confidence predictions

        box = prediction["boxes"][i].cpu().numpy()
        label = prediction["labels"][i].item()
        x_min, y_min, x_max, y_max = box

        # Draw bounding box
        rect = patches.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, linewidth=2, edgecolor='r', facecolor='none')
        ax.add_patch(rect)

        # Add label
        class_name = CLASS_MAP[label]
        plt.text(x_min, y_min - 5, f"{class_name} ({score:.2f})", color='r', fontsize=10, bbox=dict(facecolor='white', alpha=0.5))

    plt.axis("off")
    plt.show()

# ✅ Step 7: Run Inference and Display Predictions
for img_path in selected_images:
    print(f"🔍 Evaluating: {os.path.basename(img_path)}")
    img, pred = run_inference(img_path)
    draw_boxes(img, pred)


ModuleNotFoundError: No module named 'torch'